# Training Framework

## 1. Imports


In [2]:
# ==========================================================
# Standard Library
# ==========================================================
from dataclasses import dataclass, field
from datetime import datetime
from enum import Enum
from pathlib import Path
from typing import Any, Dict, List, Optional

# ==========================================================
# Third-Party Libraries
# ==========================================================
import numpy as np
import pandas as pd
import torch

# ==========================================================
# Hugging Face
# ==========================================================
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

# ==========================================================
# PEFT
# ==========================================================
from peft import (
    LoraConfig,
    IA3Config,
    TaskType,
    get_peft_model,
)

# ==========================================================
# Metrics
# ==========================================================
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
)

# ==========================================================
# Utilities
# ==========================================================
from torch.utils.data import DataLoader
from datasets import Dataset

In [3]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [4]:
try:
    import torchao
    print("TorchAO:", torchao.__version__)
except ImportError:
    print("✓ TorchAO not installed")

✓ TorchAO not installed


### Enumerations



* **PEFTMethod**
* **Language**
* **DatasetSplit**
* **ExperimentStatus**
* **DeviceType**

In [5]:
class PEFTMethod(Enum):
    FFT = "fft"
    LORA = "lora"
    DORA = "dora"
    IA3 = "ia3"

In [6]:
class Language(Enum):
    HINDI = "hi"
    TELUGU = "te"

In [7]:
class DatasetSplit(Enum):
    TRAIN = "train"
    VALIDATION = "validation"
    TEST = "test"

In [8]:
class ExperimentStatus(Enum):
    PENDING = "pending"
    RUNNING = "running"
    COMPLETED = "completed"
    FAILED = "failed"

In [9]:
class DeviceType(Enum):
    CPU = "cpu"
    CUDA = "cuda"

In [10]:
# ==========================================================
# Optimizer Types
# ==========================================================

class OptimizerType(Enum):
    """Supported optimizers."""

    ADAMW = "adamw"
    ADAM = "adam"
    SGD = "sgd"

In [11]:
# ==========================================================
# Scheduler Types
# ==========================================================

class SchedulerType(Enum):
    """Supported learning-rate schedulers."""

    LINEAR = "linear"
    COSINE = "cosine"
    CONSTANT = "constant"

## 2. Global Configuration


### ExperimentConfig

* **ExperimentInfo**
* **DatasetConfig**
* **ModelConfig**
* **PEFTConfig**
* **TrainingConfig**
* **RuntimeConfig**
* **OutputConfig**

Base Configuration Dataclasses

In [12]:
# ==========================================================
# Configuration Dataclasses
# ==========================================================

@dataclass(frozen=True)
class ExperimentInfo:
    """Metadata describing a single experiment."""

    name: str
    description: str
    seed: int
    timestamp: datetime
    status: ExperimentStatus = ExperimentStatus.PENDING


@dataclass(frozen=True)
class DatasetConfig:
    """
    Dataset configuration.
    """

    dataset_name: str

    language: Language

    budget: str

    train_path: Path
    validation_path: Path
    test_path: Path

    # Task Schema
    text_columns: tuple[str, ...]
    label_column: str

    # Optional metadata
    metadata_columns: tuple[str, ...] = ()


@dataclass(frozen=True)
class ModelConfig:
    """Backbone model configuration."""

    backbone: str
    num_labels: int
    max_length: int
    problem_type: str = "single_label_classification"

Training & Runtime Configuration

In [13]:
@dataclass(frozen=True)
class PEFTConfig:
    """PEFT configuration."""

    method: PEFTMethod
    parameters: Dict[str, Any]


# ==========================================================
# Training Configuration
# ==========================================================

@dataclass(frozen=True)
class TrainingConfig:
    """Training hyperparameters."""

    epochs: int

    batch_size: int

    learning_rate: float

    optimizer: OptimizerType

    scheduler: SchedulerType

    weight_decay: float

    warmup_ratio: float


@dataclass(frozen=True)
class RuntimeConfig:
    """Runtime configuration."""

    device: DeviceType
    fp16: bool

    gradient_accumulation: int
    num_workers: int

Output & Master Configuration

In [14]:
# ==========================================================
# Output Configuration
# ==========================================================

@dataclass(frozen=True)
class OutputConfig:
    """
    Configuration for experiment outputs.

    All experiment-specific directories are derived automatically
    from the root directory by DirectoryManager.
    """

    root_dir: Path

    save_best_only: bool = True


# ==========================================================
# Master Configuration
# ==========================================================

@dataclass(frozen=True)
class ExperimentConfig:
    """
    Root configuration object shared across the training harness.
    """

    experiment: ExperimentInfo
    dataset: DatasetConfig
    model: ModelConfig
    peft: PEFTConfig
    training: TrainingConfig
    runtime: RuntimeConfig
    output: OutputConfig

In [15]:
# ==========================================================
# Configuration Validator
# ==========================================================

from pathlib import Path


class ConfigurationValidator:
    """
    Validate experiment configuration before execution.
    """

    @classmethod
    def validate(
        cls,
        config: ExperimentConfig,
    ) -> None:
        """
        Validate all configuration sections.
        """

        cls._validate_experiment(config.experiment)
        cls._validate_dataset(config.dataset)
        cls._validate_model(config.model)
        cls._validate_peft(config.peft)
        cls._validate_training(config.training)
        cls._validate_runtime(config.runtime)
        cls._validate_output(config.output)

    # ------------------------------------------------------
    # Experiment
    # ------------------------------------------------------

    @staticmethod
    def _validate_experiment(
        experiment: ExperimentInfo,
    ) -> None:

        if not experiment.name.strip():
            raise ValueError(
                "Experiment name cannot be empty."
            )

        if experiment.seed < 0:
            raise ValueError(
                "Seed must be non-negative."
            )

    # ------------------------------------------------------
    # Dataset
    # ------------------------------------------------------

    @staticmethod
    def _validate_dataset(
        dataset: DatasetConfig,
    ) -> None:

        valid_budgets = {
            "50",
            "100",
            "500",
            "1000",
            "full",
        }

        if dataset.budget not in valid_budgets:
            raise ValueError(
                f"Invalid budget: {dataset.budget}"
            )

        required_paths = (
            dataset.train_path,
            dataset.validation_path,
            dataset.test_path,
        )

        for path in required_paths:

            if not isinstance(path, Path):
                raise TypeError(
                    "Dataset paths must be pathlib.Path objects."
                )

            if not path.exists():
                raise FileNotFoundError(
                    f"Dataset file not found: {path}"
                )

    # ------------------------------------------------------
    # Model
    # ------------------------------------------------------

    @staticmethod
    def _validate_model(
        model: ModelConfig,
    ) -> None:

        if model.num_labels < 2:
            raise ValueError(
                "num_labels must be at least 2."
            )

        if model.max_length <= 0:
            raise ValueError(
                "max_length must be greater than zero."
            )

    # ------------------------------------------------------
    # PEFT
    # ------------------------------------------------------

    @staticmethod
    def _validate_peft(
        peft: PEFTConfig,
    ) -> None:

        if not isinstance(peft.parameters, dict):
            raise TypeError(
                "PEFT parameters must be a dictionary."
            )

    # ------------------------------------------------------
    # Training
    # ------------------------------------------------------

    @staticmethod
    def _validate_training(
        training: TrainingConfig,
    ) -> None:

        if training.epochs <= 0:
            raise ValueError(
                "epochs must be greater than zero."
            )

        if training.batch_size <= 0:
            raise ValueError(
                "batch_size must be greater than zero."
            )

        if training.learning_rate <= 0:
            raise ValueError(
                "learning_rate must be greater than zero."
            )

        if not isinstance(
            training.optimizer,
            OptimizerType,
            ):
            raise TypeError(
                "optimizer must be an OptimizerType."
            )
            
        if not isinstance(
            training.scheduler,
            SchedulerType,
            ):
            raise TypeError(
                "scheduler must be a SchedulerType."
            )
        if training.weight_decay < 0:
            raise ValueError(
                "weight_decay cannot be negative."
            )

        if not 0.0 <= training.warmup_ratio <= 1.0:
            raise ValueError(
                "warmup_ratio must be between 0 and 1."
            )

    # ------------------------------------------------------
    # Runtime
    # ------------------------------------------------------

    @staticmethod
    def _validate_runtime(
        runtime: RuntimeConfig,
    ) -> None:

        if runtime.gradient_accumulation <= 0:
            raise ValueError(
                "gradient_accumulation must be greater than zero."
            )

        if runtime.num_workers < 0:
            raise ValueError(
                "num_workers cannot be negative."
            )

    # ------------------------------------------------------
    # Output
    # ------------------------------------------------------

    @staticmethod
    def _validate_output(
        output: OutputConfig,
    ) -> None:

        if not isinstance(output.root_dir, Path):
            raise TypeError(
                "root_dir must be a pathlib.Path object."
            )

        if not str(output.root_dir).strip():
            raise ValueError(
                "Output root directory cannot be empty."
            )

In [16]:
# ==========================================================
# Seed Manager
# ==========================================================

import os
import random

class SeedManager:
    """Centralized random seed management."""

    @staticmethod
    def set_seed(seed: int) -> int:
        """
        Set random seeds for reproducibility.

        Parameters
        ----------
        seed : int
            Random seed.

        Returns
        -------
        int
            Applied seed.
        """

        random.seed(seed)

        np.random.seed(seed)

        os.environ["PYTHONHASHSEED"] = str(seed)

        torch.manual_seed(seed)

        if torch.cuda.is_available():

            torch.cuda.manual_seed(seed)

            torch.cuda.manual_seed_all(seed)

        torch.backends.cudnn.deterministic = True

        torch.backends.cudnn.benchmark = False

        return seed

In [17]:
# ==========================================================
# Device Manager
# ==========================================================

class DeviceManager:
    """Device discovery and runtime management."""

    @staticmethod
    def get_device(preferred: Optional[DeviceType] = None) -> torch.device:
        """
        Resolve the execution device.

        Priority:
            1. User preference (if available)
            2. CUDA
            3. CPU
        """

        if preferred == DeviceType.CPU:
            return torch.device("cpu")

        if torch.cuda.is_available():
            return torch.device("cuda")

        return torch.device("cpu")


    @staticmethod
    def get_device_info(device: torch.device) -> Dict[str, Any]:
        """
        Collect runtime information about the selected device.
        """

        info = {
            "device": str(device),
            "device_name": None,
            "num_gpus": 0,
            "gpu_memory_gb": None,
            "mixed_precision": False,
        }

        if device.type == "cuda":

            props = torch.cuda.get_device_properties(device)

            info["device_name"] = props.name
            info["num_gpus"] = torch.cuda.device_count()
            info["gpu_memory_gb"] = round(
                props.total_memory / (1024**3), 2
            )
            info["mixed_precision"] = True

        return info

Experiment Paths + Naming + Directory

In [18]:
# ==========================================================
# Experiment Paths & Directory Management
# ==========================================================

from dataclasses import dataclass
from datetime import datetime
from pathlib import Path


# ----------------------------------------------------------
# Experiment Paths
# ----------------------------------------------------------

@dataclass(frozen=True)
class ExperimentPaths:
    """
    Container for experiment-specific directory paths.
    """

    root: Path
    checkpoints: Path
    logs: Path
    metrics: Path
    plots: Path
    predictions: Path


# ----------------------------------------------------------
# Experiment Naming
# ----------------------------------------------------------

class ExperimentNaming:
    """
    Generate standardized experiment names.
    """

    @staticmethod
    def generate(config: ExperimentConfig) -> str:
        """
        Generate a deterministic experiment name.
        """

        return (
            f"{config.peft.method.value}_"
            f"{config.dataset.language.value}_"
            f"budget{config.dataset.budget}_"
            f"seed{config.experiment.seed}"
        )

    @staticmethod
    def generate_with_timestamp(
        config: ExperimentConfig,
    ) -> str:
        """
        Generate an experiment name with a timestamp suffix.
        """

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

        return (
            f"{ExperimentNaming.generate(config)}"
            f"_{timestamp}"
        )


# ----------------------------------------------------------
# Directory Manager
# ----------------------------------------------------------

class DirectoryManager:
    """
    Create and manage experiment directory structure.
    """

    STANDARD_DIRECTORIES = (
        "checkpoints",
        "logs",
        "metrics",
        "plots",
        "predictions",
    )

    @staticmethod
    def _create_directory(path: Path) -> Path:
        """
        Create a directory if it does not already exist.
        """

        path.mkdir(
            parents=True,
            exist_ok=True,
        )

        return path

    @classmethod
    def prepare(
        cls,
        config: ExperimentConfig,
        use_timestamp: bool = False,
    ) -> ExperimentPaths:
        """
        Create the complete experiment directory structure.

        Parameters
        ----------
        config : ExperimentConfig
            Experiment configuration.

        use_timestamp : bool, default=False
            Append a timestamp to the experiment name.

        Returns
        -------
        ExperimentPaths
            Paths to all experiment directories.
        """

        # --------------------------------------------------
        # Root directory
        # --------------------------------------------------

        root_dir = cls._create_directory(
            config.output.root_dir
        )

        # --------------------------------------------------
        # Experiment directory
        # --------------------------------------------------

        experiment_name = (
            ExperimentNaming.generate_with_timestamp(config)
            if use_timestamp
            else ExperimentNaming.generate(config)
        )

        experiment_dir = cls._create_directory(
            root_dir / experiment_name
        )

        # --------------------------------------------------
        # Standard experiment directories
        # --------------------------------------------------

        directories = {
            name: cls._create_directory(
                experiment_dir / name
            )
            for name in cls.STANDARD_DIRECTORIES
        }

        # --------------------------------------------------
        # Return strongly typed paths
        # --------------------------------------------------

        return ExperimentPaths(
            root=experiment_dir,
            checkpoints=directories["checkpoints"],
            logs=directories["logs"],
            metrics=directories["metrics"],
            plots=directories["plots"],
            predictions=directories["predictions"],
        )

    @staticmethod
    def cleanup_empty(path: Path) -> None:
        """
        Remove an empty experiment directory.
        """

        if path.exists() and not any(path.iterdir()):
            path.rmdir()

ExperimentContext

In [19]:
# ==========================================================
# Experiment Context
# ==========================================================

@dataclass(frozen=True)
class ExperimentContext:
    """
    Runtime context shared across the entire experiment.
    """

    config: ExperimentConfig

    paths: ExperimentPaths

    device: torch.device

    runtime_info: Dict[str, Any]

    start_time: datetime

## Experiment Manager

In [20]:
# ==========================================================
# Experiment Manager
# ==========================================================

class ExperimentManager:
    """
    Factory for creating experiment configurations.

    This class centralizes all default experiment settings,
    allowing experiments to be created with minimal code.
    """

    # ------------------------------------------------------
    # Public Builder
    # ------------------------------------------------------

    @classmethod
    def create(
        cls,
        *,
        experiment_name: str,
        description: str,
        language: Language,
        budget: str,
        method: PEFTMethod,
        train_path: Path,
        validation_path: Path,
        test_path: Path,
        output_dir: Path,
        epochs: int = 10,
        batch_size: int = 4,
        learning_rate: float = 2e-5,
        weight_decay: float = 0.01,
        warmup_ratio: float = 0.10,
        seed: int = 42,
    ) -> ExperimentConfig:
        """
        Create a complete experiment configuration.
        """

        return ExperimentConfig(

            experiment=ExperimentInfo(

                name=experiment_name,

                description=description,

                seed=seed,

                timestamp=datetime.now(),

            ),

            dataset=DatasetConfig(

                dataset_name="IndicXNLI",

                language=language,

                budget=budget,

                train_path=train_path,

                validation_path=validation_path,

                test_path=test_path,

                text_columns=(
                    "premise",
                    "hypothesis",
                ),

                label_column="label",

            ),

            model=ModelConfig(

                backbone="xlm-roberta-base",

                num_labels=3,

                max_length=128,

            ),

            peft=PEFTConfig(

                method=method,

                parameters=cls._peft_parameters(
                    method,
                ),

            ),

            training=TrainingConfig(

                epochs=epochs,

                batch_size=batch_size,

                learning_rate=learning_rate,

                optimizer=OptimizerType.ADAMW,

                scheduler=SchedulerType.LINEAR,

                weight_decay=weight_decay,

                warmup_ratio=warmup_ratio,

            ),

            runtime=RuntimeConfig(

                device=DeviceType.CUDA,

                fp16=True,

                gradient_accumulation=1,

                num_workers=2,

            ),

            output=OutputConfig(

                root_dir=output_dir,

                save_best_only=True,

            ),

        )

    # ------------------------------------------------------
    # PEFT Defaults
    # ------------------------------------------------------

    @staticmethod
    def _peft_parameters(
        method: PEFTMethod,
    ) -> dict:
        """
        Return default parameters for each PEFT method.
        """

        if method == PEFTMethod.FFT:

            return {}

        if method == PEFTMethod.LORA:

            return {

                "r": 8,

                "lora_alpha": 16,

                "lora_dropout": 0.10,

                "bias": "none",

                "target_modules": [

                    "query",

                    "value",

                ],

            }

        if method == PEFTMethod.DORA:

            return {

                "r": 8,

                "lora_alpha": 16,

                "lora_dropout": 0.10,

                "bias": "none",

                "use_dora": True,

                "target_modules": [

                    "query",

                    "value",

                ],

            }

        if method == PEFTMethod.IA3:

            return {

                "target_modules": [

                    "key",

                    "value",

                    "output.dense",

                ],

            }

        raise ValueError(
            f"Unsupported PEFT method: {method}"
        )

## ModelFactory Implementation

In [21]:
import torch

from transformers import (
    AutoModelForSequenceClassification,
)

In [22]:
# ==========================================================
# Model Factory
# ==========================================================

class ModelBuildError(Exception):
    """Raised when model construction fails."""
    pass


@dataclass(frozen=True)
class ModelBundle:
    """
    Fully initialized model and metadata.
    """

    model: torch.nn.Module

    total_parameters: int

    trainable_parameters: int

    trainable_percentage: float


class ModelFactory:
    """
    Factory responsible for constructing backbone and PEFT models.
    """

    # ------------------------------------------------------
    # Backbone Loading
    # ------------------------------------------------------

    @classmethod
    def _load_backbone(
        cls,
        config: ModelConfig,
    ) -> torch.nn.Module:
        """
        Load Hugging Face sequence classification model.
        """

        try:

            print(f"Loading backbone: {config.backbone}")

            model = AutoModelForSequenceClassification.from_pretrained(
                config.backbone,
                num_labels=config.num_labels,
                problem_type=config.problem_type,
            )

            return model

        except Exception as e:

            raise ModelBuildError(
                f"Failed to load backbone '{config.backbone}'."
            ) from e

    # ------------------------------------------------------
    # Parameter Statistics
    # ------------------------------------------------------

    @staticmethod
    def _count_trainable_parameters(
        model: torch.nn.Module,
    ) -> tuple[int, int, float]:
        """
        Calculate parameter statistics.
        """

        total_parameters = sum(
            parameter.numel()
            for parameter in model.parameters()
        )

        trainable_parameters = sum(
            parameter.numel()
            for parameter in model.parameters()
            if parameter.requires_grad
        )

        trainable_percentage = (
            trainable_parameters / total_parameters * 100
            if total_parameters > 0
            else 0.0
        )

        return (
            total_parameters,
            trainable_parameters,
            trainable_percentage,
        )

    # ------------------------------------------------------
    # PEFT Builders
    # ------------------------------------------------------

    @classmethod
    def _apply_fft(
        cls,
        model: torch.nn.Module,
        context: ExperimentContext,
    ) -> torch.nn.Module:
        """
        Full Fine-Tuning.
        """

        return model


    @classmethod
    def _apply_lora(
        cls,
        model: torch.nn.Module,
        context: ExperimentContext,
    ) -> torch.nn.Module:
        """
        Apply LoRA.
        """

        peft_config = LoraConfig(
            task_type=TaskType.SEQ_CLS,
            **context.config.peft.parameters,
        )

        return get_peft_model(
            model,
            peft_config,
        )


    @classmethod
    def _apply_dora(
        cls,
        model: torch.nn.Module,
        context: ExperimentContext,
    ) -> torch.nn.Module:
        """
        Apply DoRA.
        """

        peft_config = LoraConfig(
            task_type=TaskType.SEQ_CLS,
            **context.config.peft.parameters,
        )

        return get_peft_model(
            model,
            peft_config,
        )


    @classmethod
    def _apply_ia3(
        cls,
        model: torch.nn.Module,
        context: ExperimentContext,
    ) -> torch.nn.Module:
        """
        Apply IA3.
        """

        peft_config = IA3Config(
            task_type=TaskType.SEQ_CLS,
            **context.config.peft.parameters,
        )

        return get_peft_model(
            model,
            peft_config,
        )

        # ------------------------------------------------------
    # Model Validation
    # ------------------------------------------------------

    @classmethod
    def _validate_model(
        cls,
        model: torch.nn.Module,
        context: ExperimentContext,
    ) -> None:
        """
        Validate the constructed model.
        """

        if model is None:
            raise ModelBuildError(
                "Model construction returned None."
            )

        if not isinstance(model, torch.nn.Module):
            raise ModelBuildError(
                "Constructed object is not a torch.nn.Module."
            )

        if model.config.num_labels != context.config.model.num_labels:
            raise ModelBuildError(
                "Model num_labels does not match configuration."
            )

        _, trainable, _ = cls._count_trainable_parameters(
            model
        )

        if trainable == 0:
            raise ModelBuildError(
                "No trainable parameters detected."
            )

    # ------------------------------------------------------
    # Public Builder
    # ------------------------------------------------------

    @classmethod
    def build(
        cls,
        context: ExperimentContext,
    ) -> ModelBundle:
        """
        Build, validate and package a model.
        """

        # ----------------------------------------------
        # Load Backbone
        # ----------------------------------------------

        model = cls._load_backbone(
            context.config.model
        )

        # ----------------------------------------------
        # Apply PEFT
        # ----------------------------------------------

        method = context.config.peft.method

        # Normalize enum or string to a plain string
        method = getattr(method, "value", method)

        if method == "fft":

            model = cls._apply_fft(
                model,
                context,
            )

        elif method == "lora":

            model = cls._apply_lora(
                model,
                context,
            )

        elif method == "dora":

            model = cls._apply_dora(
                model,
                context,
            )

        elif method == "ia3":

            model = cls._apply_ia3(
            model,
            context,
            )

        else:

            raise ModelBuildError(
                f"Unsupported PEFT method: {method}"
            )
        
        
          
            

        # ----------------------------------------------
        # Validate
        # ----------------------------------------------

        cls._validate_model(
            model,
            context,
        )

        # ----------------------------------------------
        # Parameter Statistics
        # ----------------------------------------------

        (
            total,
            trainable,
            percentage,
        ) = cls._count_trainable_parameters(
            model
        )

        # ----------------------------------------------
        # Package
        # ----------------------------------------------

        return ModelBundle(
            model=model,
            total_parameters=total,
            trainable_parameters=trainable,
            trainable_percentage=percentage,
        )

In [23]:
import torch
import transformers
import peft

print(torch.__version__)
print(transformers.__version__)
print(peft.__version__)

try:
    import torchao
    print(torchao.__version__)
except Exception as e:
    print(e)

2.10.0+cu128
5.0.0
0.19.1
No module named 'torchao'


## DatasetManager

In [24]:
# ==========================================================
# Dataset Manager
# ==========================================================

class DatasetError(Exception):
    """Raised when dataset loading fails."""
    pass


class DatasetManager:
    """
    Responsible for loading experiment datasets.
    """

    @classmethod
    def load_train(
        cls,
        context: ExperimentContext,
    ) -> Dataset:

        return cls._load_parquet(
            context.config.dataset.train_path
        )


    @classmethod
    def load_validation(
        cls,
        context: ExperimentContext,
    ) -> Dataset:

        return cls._load_parquet(
            context.config.dataset.validation_path
        )


    @classmethod
    def load_test(
        cls,
        context: ExperimentContext,
    ) -> Dataset:

        return cls._load_parquet(
            context.config.dataset.test_path
        )


    @classmethod
    def load_all(
        cls,
        context: ExperimentContext,
    ) -> tuple[Dataset, Dataset, Dataset]:

        train = cls.load_train(context)

        validation = cls.load_validation(context)

        test = cls.load_test(context)

        return train, validation, test


    @staticmethod
    def _load_parquet(
        path: Path,
    ) -> Dataset:
        """
        Load a parquet file into a Hugging Face Dataset.
        """

        try:

            dataframe = pd.read_parquet(path)

            return Dataset.from_pandas(
                dataframe,
                preserve_index=False,
            )

        except Exception as e:

            raise DatasetError(
                f"Failed to load dataset: {path}"
            ) from e

## TokenizerManager

In [25]:
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    DataCollatorWithPadding,
)
from torch.utils.data import DataLoader


# ==========================================================
# Tokenizer Manager
# ==========================================================

class TokenizationError(Exception):
    """Raised when tokenizer initialization or dataset tokenization fails."""
    pass


class TokenizerManager:
    """
    Responsible for tokenizer initialization, dataset tokenization,
    dataset preparation, and DataLoader creation.

    Responsibilities
    ----------------
    • Load tokenizer
    • Tokenize datasets
    • Prepare datasets for PyTorch
    • Create DataLoaders
    • Prepare the complete data pipeline

    This module intentionally does NOT:
    • Load datasets
    • Build models
    • Train models
    """

    # ------------------------------------------------------
    # Tokenizer
    # ------------------------------------------------------

    @classmethod
    def load_tokenizer(
        cls,
        context: "ExperimentContext",
    ) -> AutoTokenizer:
        """
        Load the Hugging Face tokenizer.
        """

        try:

            return AutoTokenizer.from_pretrained(
                context.config.model.backbone,
            )

        except Exception as e:

            raise TokenizationError(
                f"Unable to load tokenizer "
                f"{context.config.model.backbone}"
            ) from e

    # ------------------------------------------------------
    # Dataset Tokenization
    # ------------------------------------------------------

    @classmethod
    def tokenize_dataset(
        cls,
        dataset: Dataset,
        tokenizer: AutoTokenizer,
        context: "ExperimentContext",
    ) -> Dataset:
        """
        Tokenize a Hugging Face dataset.
        """

        text_columns = (
            context.config.dataset.text_columns
        )

        max_length = (
            context.config.model.max_length
        )

        def tokenize_function(batch):

            if len(text_columns) == 1:

                return tokenizer(
                    batch[text_columns[0]],
                    truncation=True,
                    max_length=max_length,
                )

            elif len(text_columns) == 2:

                return tokenizer(
                    batch[text_columns[0]],
                    batch[text_columns[1]],
                    truncation=True,
                    max_length=max_length,
                )

            raise TokenizationError(
                "Unsupported number of text columns."
            )

        return dataset.map(
            tokenize_function,
            batched=True,
            desc="Tokenizing dataset",
        )

    # ------------------------------------------------------
    # Dataset Preparation
    # ------------------------------------------------------

    @classmethod
    def prepare_dataset(
        cls,
        dataset: Dataset,
        context: "ExperimentContext",
    ) -> Dataset:
        """
        Convert a tokenized dataset to PyTorch format.
        """

        label_column = (
            context.config.dataset.label_column
        )

        if (
            label_column != "labels"
            and label_column in dataset.column_names
        ):
            dataset = dataset.rename_column(
                label_column,
                "labels",
            )

        columns = [
            "input_ids",
            "attention_mask",
            "labels",
        ]

        if "token_type_ids" in dataset.column_names:
            columns.append(
                "token_type_ids"
            )

        dataset.set_format(
            type="torch",
            columns=columns,
        )

        return dataset

    # ------------------------------------------------------
    # DataLoader
    # ------------------------------------------------------

    @classmethod
    def create_dataloader(
        cls,
        dataset: Dataset,
        tokenizer: AutoTokenizer,
        shuffle: bool,
        batch_size: int,
    ) -> DataLoader:
        """
        Create a PyTorch DataLoader with dynamic padding.
        """

        collator = DataCollatorWithPadding(
            tokenizer=tokenizer,
        )

        return DataLoader(
            dataset,
            batch_size=batch_size,
            shuffle=shuffle,
            collate_fn=collator,
        )

    # ------------------------------------------------------
    # Complete Preparation Pipeline
    # ------------------------------------------------------

    @classmethod
    def prepare(
        cls,
        train: Dataset,
        validation: Dataset,
        test: Dataset,
        context: "ExperimentContext",
    ) -> tuple[
        AutoTokenizer,
        DataLoader,
        DataLoader,
        DataLoader,
    ]:
        """
        Prepare tokenizer, datasets, and dataloaders.
        """

        tokenizer = cls.load_tokenizer(
            context,
        )

        train = cls.tokenize_dataset(
            train,
            tokenizer,
            context,
        )

        validation = cls.tokenize_dataset(
            validation,
            tokenizer,
            context,
        )

        test = cls.tokenize_dataset(
            test,
            tokenizer,
            context,
        )

        train = cls.prepare_dataset(
            train,
            context,
        )

        validation = cls.prepare_dataset(
            validation,
            context,
        )

        test = cls.prepare_dataset(
            test,
            context,
        )

        batch_size = (
            context.config.training.batch_size
        )

        train_loader = cls.create_dataloader(
            train,
            tokenizer,
            shuffle=True,
            batch_size=batch_size,
        )

        validation_loader = cls.create_dataloader(
            validation,
            tokenizer,
            shuffle=False,
            batch_size=batch_size,
        )

        test_loader = cls.create_dataloader(
            test,
            tokenizer,
            shuffle=False,
            batch_size=batch_size,
        )

        return (
            tokenizer,
            train_loader,
            validation_loader,
            test_loader,
        )

## Training Manager

In [26]:
# ==========================================================
# Training Manager
# ==========================================================

from dataclasses import dataclass

import torch

from torch.optim import (
    Optimizer,
    Adam,
    AdamW,
    SGD,
)

from torch.optim.lr_scheduler import LRScheduler

from transformers import (
    get_linear_schedule_with_warmup,
    get_cosine_schedule_with_warmup,
    get_constant_schedule_with_warmup,
)


# ==========================================================
# Training Exceptions
# ==========================================================

class TrainingError(Exception):
    """
    Raised when training utilities cannot be constructed.
    """
    pass


# ==========================================================
# Training Bundle
# ==========================================================

@dataclass(frozen=True)
class TrainingBundle:
    """
    Bundle containing all training utilities.
    """

    optimizer: Optimizer

    scheduler: LRScheduler

    scaler: torch.cuda.amp.GradScaler | None


# ==========================================================
# Training Manager
# ==========================================================

class TrainingManager:
    """
    Factory responsible for constructing all training
    utilities required by the Trainer.

    Responsibilities
    ----------------
    • Create optimizer
    • Create scheduler
    • Create gradient scaler
    • Validate training utilities
    • Package everything into a TrainingBundle
    """

    # ------------------------------------------------------
    # Optimizer Builders
    # ------------------------------------------------------

    @classmethod
    def _build_adamw(
        cls,
        model: torch.nn.Module,
        context: ExperimentContext,
    ) -> Optimizer:
        """
        Construct AdamW optimizer.
        """

        return AdamW(
            model.parameters(),
            lr=context.config.training.learning_rate,
            weight_decay=context.config.training.weight_decay,
        )


    @classmethod
    def _build_adam(
        cls,
        model: torch.nn.Module,
        context: ExperimentContext,
    ) -> Optimizer:
        """
        Construct Adam optimizer.
        """

        return Adam(
            model.parameters(),
            lr=context.config.training.learning_rate,
            weight_decay=context.config.training.weight_decay,
        )


    @classmethod
    def _build_sgd(
        cls,
        model: torch.nn.Module,
        context: ExperimentContext,
    ) -> Optimizer:
        """
        Construct SGD optimizer.
        """

        return SGD(
            model.parameters(),
            lr=context.config.training.learning_rate,
            weight_decay=context.config.training.weight_decay,
        )


    # ------------------------------------------------------
    # Public Optimizer Factory
    # ------------------------------------------------------

    @classmethod
    def create_optimizer(
        cls,
        model: torch.nn.Module,
        context: ExperimentContext,
    ) -> Optimizer:
        """
        Create optimizer from configuration.
        """

        optimizer = context.config.training.optimizer

        if isinstance(optimizer, str):
            optimizer = OptimizerType(optimizer)

        if optimizer == OptimizerType.ADAMW:

            return cls._build_adamw(
                model,
                context,
            )

        elif optimizer == OptimizerType.ADAM:

            return cls._build_adam(
                model,
                context,
            )

        elif optimizer == OptimizerType.SGD:

            return cls._build_sgd(
                model,
                context,
            )

        raise TrainingError(
            f"Unsupported optimizer: {optimizer}"
        )    # ------------------------------------------------------
    # Scheduler Builders
    # ------------------------------------------------------

    @classmethod
    def _build_linear_scheduler(
        cls,
        optimizer: Optimizer,
        train_loader,
        context: ExperimentContext,
    ) -> LRScheduler:
        """
        Construct linear warmup scheduler.
        """

        total_steps = (
            len(train_loader)
            * context.config.training.epochs
        )

        warmup_steps = int(
            total_steps
            * context.config.training.warmup_ratio
        )

        return get_linear_schedule_with_warmup(
            optimizer=optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps,
        )


    @classmethod
    def _build_cosine_scheduler(
        cls,
        optimizer: Optimizer,
        train_loader,
        context: ExperimentContext,
    ) -> LRScheduler:
        """
        Construct cosine warmup scheduler.
        """

        total_steps = (
            len(train_loader)
            * context.config.training.epochs
        )

        warmup_steps = int(
            total_steps
            * context.config.training.warmup_ratio
        )

        return get_cosine_schedule_with_warmup(
            optimizer=optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=total_steps,
        )


    @classmethod
    def _build_constant_scheduler(
        cls,
        optimizer: Optimizer,
        train_loader,
        context: ExperimentContext,
    ) -> LRScheduler:
        """
        Construct constant scheduler with warmup.
        """

        total_steps = (
            len(train_loader)
            * context.config.training.epochs
        )

        warmup_steps = int(
            total_steps
            * context.config.training.warmup_ratio
        )

        return get_constant_schedule_with_warmup(
            optimizer=optimizer,
            num_warmup_steps=warmup_steps,
        )


    # ------------------------------------------------------
    # Public Scheduler Factory
    # ------------------------------------------------------

    @classmethod
    def create_scheduler(
        cls,
        optimizer: Optimizer,
        train_loader,
        context: ExperimentContext,
    ) -> LRScheduler:
        """
        Create scheduler from configuration.
        """

        scheduler = context.config.training.scheduler

        if isinstance(scheduler, str):
            scheduler = SchedulerType(scheduler)

        if scheduler == SchedulerType.LINEAR:

            return cls._build_linear_scheduler(
                optimizer,
                train_loader,
                context,
            )

        elif scheduler == SchedulerType.COSINE:

            return cls._build_cosine_scheduler(
                optimizer,
                train_loader,
                context,
            )

        elif scheduler == SchedulerType.CONSTANT:

            return cls._build_constant_scheduler(
                optimizer,
                train_loader,
                context,
            )

        raise TrainingError(
            f"Unsupported scheduler: {scheduler}"
        )


    # ------------------------------------------------------
    # Gradient Scaler
    # ------------------------------------------------------

    @classmethod
    def create_grad_scaler(
        cls,
        context: ExperimentContext,
    ) -> torch.cuda.amp.GradScaler | None:
        """
        Create GradScaler for mixed precision training.
        """

        if context.config.runtime.fp16:

            return torch.cuda.amp.GradScaler()

        return None


    # ------------------------------------------------------
    # Validation
    # ------------------------------------------------------

    @classmethod
    def _validate(
        cls,
        bundle: TrainingBundle,
    ) -> None:
        """
        Validate constructed training utilities.
        """

        if bundle.optimizer is None:
            raise TrainingError(
                "Optimizer construction failed."
            )

        if bundle.scheduler is None:
            raise TrainingError(
                "Scheduler construction failed."
            )

        if not isinstance(
            bundle.optimizer,
            Optimizer,
        ):
            raise TrainingError(
                "Invalid optimizer instance."
            )

        if not isinstance(
            bundle.scheduler,
            LRScheduler,
        ):
            raise TrainingError(
                "Invalid scheduler instance."
            )    # ------------------------------------------------------
    # Public Builder
    # ------------------------------------------------------

    @classmethod
    def prepare(
        cls,
        model: torch.nn.Module,
        train_loader,
        context: ExperimentContext,
    ) -> TrainingBundle:
        """
        Construct all training utilities.

        Parameters
        ----------
        model : torch.nn.Module
            Model to optimize.

        train_loader : DataLoader
            Training DataLoader.

        context : ExperimentContext
            Experiment configuration.

        Returns
        -------
        TrainingBundle
            Optimizer, scheduler and gradient scaler.
        """

        # ----------------------------------------------
        # Optimizer
        # ----------------------------------------------

        optimizer = cls.create_optimizer(
            model,
            context,
        )

        # ----------------------------------------------
        # Scheduler
        # ----------------------------------------------

        scheduler = cls.create_scheduler(
            optimizer,
            train_loader,
            context,
        )

        # ----------------------------------------------
        # Gradient Scaler
        # ----------------------------------------------

        scaler = cls.create_grad_scaler(
            context,
        )

        # ----------------------------------------------
        # Package
        # ----------------------------------------------

        bundle = TrainingBundle(
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
        )

        # ----------------------------------------------
        # Validate
        # ----------------------------------------------

        cls._validate(bundle)

        return bundle

## TRAINER

In [27]:
# ==========================================================
# Trainer
# ==========================================================

import time
from dataclasses import dataclass

import torch

from torch.utils.data import DataLoader

from tqdm.auto import tqdm


# ==========================================================
# Trainer Exceptions
# ==========================================================

class TrainerError(Exception):
    """
    Raised when training fails.
    """
    pass


# ==========================================================
# Epoch Result
# ==========================================================

@dataclass(frozen=True)
class EpochResult:
    """
    Result of a single epoch.
    """

    loss: float


# ==========================================================
# Training History
# ==========================================================

@dataclass
class TrainingHistory:
    """
    Stores complete training history.
    """

    train_loss: list[float]

    validation_loss: list[float]


# ==========================================================
# Trainer
# ==========================================================

class Trainer:
    """
    Responsible for model optimization.

    Responsibilities
    ----------------
    • Train batches
    • Train epochs
    • Validate epochs
    • Maintain training history

    This class intentionally does NOT

    • Build models
    • Create optimizers
    • Compute metrics
    • Save checkpoints
    """

    # ------------------------------------------------------
    # Constructor
    # ------------------------------------------------------

    def __init__(
        self,
        model: torch.nn.Module,
        training: TrainingBundle,
        train_loader: DataLoader,
        validation_loader: DataLoader,
        context: ExperimentContext,
    ):

        self.model = model

        self.optimizer = training.optimizer

        self.scheduler = training.scheduler

        self.scaler = training.scaler

        self.train_loader = train_loader

        self.validation_loader = validation_loader

        self.context = context

        self.device = context.device

    # ------------------------------------------------------
    # Helpers
    # ------------------------------------------------------

    def _move_batch_to_device(
        self,
        batch: dict,
    ) -> dict:
        """
        Move an entire batch onto the target device.
        """

        return {

            key: value.to(self.device)

            for key, value in batch.items()

        }

    # ------------------------------------------------------
    # Train Batch
    # ------------------------------------------------------

    def train_batch(
        self,
        batch: dict,
    ) -> float:
        """
        Train the model on a single batch.

        Parameters
        ----------
        batch : dict
            Batch produced by the DataLoader.

        Returns
        -------
        float
            Batch loss.
        """

        # ----------------------------------------------
        # Move Batch
        # ----------------------------------------------

        batch = self._move_batch_to_device(
            batch
        )

        # ----------------------------------------------
        # Reset Gradients
        # ----------------------------------------------

        self.optimizer.zero_grad()

        # ----------------------------------------------
        # Automatic Mixed Precision
        # ----------------------------------------------

        use_amp = (
            self.scaler is not None
        )

        device_type = self.device.type

        with torch.autocast(
            device_type=device_type,
            enabled=use_amp,
        ):

            outputs = self.model(
                **batch
            )

            loss = outputs.loss

        # ----------------------------------------------
        # Backpropagation
        # ----------------------------------------------

        if use_amp:

            self.scaler.scale(
                loss
            ).backward()

            self.scaler.step(
                self.optimizer
            )

            self.scaler.update()

        else:

            loss.backward()

            self.optimizer.step()

        # ----------------------------------------------
        # Scheduler Step
        # ----------------------------------------------

        self.scheduler.step()

        # ----------------------------------------------
        # Return Loss
        # ----------------------------------------------

        return loss.detach().item()   
    # ------------------------------------------------------
    # Train Epoch
    # ------------------------------------------------------

    def train_epoch(
        self,
    ) -> EpochResult:
        """
        Train the model for one epoch.
        """

        self.model.train()

        running_loss = 0.0

        progress_bar = tqdm(
            self.train_loader,
            desc="Training",
            leave=False,
        )

        for batch in progress_bar:

            loss = self.train_batch(
                batch,
            )

            running_loss += loss

            progress_bar.set_postfix(
                loss=f"{loss:.4f}",
            )

        average_loss = (
            running_loss
            / len(self.train_loader)
        )

        return EpochResult(
            loss=average_loss,
        )


    # ------------------------------------------------------
    # Validation Epoch
    # ------------------------------------------------------

    def validate_epoch(
        self,
    ) -> EpochResult:
        """
        Evaluate the model on the validation dataset.
        """

        self.model.eval()

        running_loss = 0.0

        progress_bar = tqdm(
            self.validation_loader,
            desc="Validation",
            leave=False,
        )

        with torch.no_grad():

            for batch in progress_bar:

                batch = self._move_batch_to_device(
                    batch,
                )

                use_amp = (
                    self.scaler is not None
                )

                with torch.autocast(
                    device_type=self.device.type,
                    enabled=use_amp,
                ):

                    outputs = self.model(
                        **batch,
                    )

                    loss = outputs.loss

                running_loss += (
                    loss.detach().item()
                )

                progress_bar.set_postfix(
                    loss=f"{loss:.4f}",
                )

        average_loss = (
            running_loss
            / len(self.validation_loader)
        )

        return EpochResult(
            loss=average_loss,
        )    # ------------------------------------------------------
    # Training Loop
    # ------------------------------------------------------

    def fit(
        self,
    ) -> TrainingHistory:
        """
        Train the model for all configured epochs.

        Returns
        -------
        TrainingHistory
            Complete training history.
        """

        history = TrainingHistory(
            train_loss=[],
            validation_loss=[],
        )

        epochs = self.context.config.training.epochs

        print("=" * 70)
        print("TRAINING")
        print("=" * 70)

        for epoch in range(epochs):

            print(
                f"\nEpoch "
                f"{epoch + 1}/{epochs}"
            )

            print("-" * 70)

            # ------------------------------------------
            # Timing Baseline
            # ------------------------------------------

            start = time.perf_counter()

            # ------------------------------------------
            # Train
            # ------------------------------------------

            train_result = self.train_epoch()

            # ------------------------------------------
            # Validation
            # ------------------------------------------

            validation_result = (
                self.validate_epoch()
            )

            # ------------------------------------------
            # Timing Stop
            # ------------------------------------------

            elapsed = time.perf_counter() - start

            # ------------------------------------------
            # Save History
            # ------------------------------------------

            history.train_loss.append(
                train_result.loss
            )

            history.validation_loss.append(
                validation_result.loss
            )

            # ------------------------------------------
            # Epoch Summary
            # ------------------------------------------

            print(
                f"Train Loss      : "
                f"{train_result.loss:.4f}"
            )

            print(
                f"Validation Loss : "
                f"{validation_result.loss:.4f}"
            )

            print(
                f"Time            : "
                f"{elapsed:.2f} sec"
            )

        print("\n" + "=" * 70)
        print("Training Complete")
        print("=" * 70)

        return history

## EVALUATOR

In [28]:
# ==========================================================
# Evaluator
# ==========================================================

from dataclasses import dataclass

import numpy as np
import torch

from torch.utils.data import DataLoader

from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
)


# ==========================================================
# Evaluation Exceptions
# ==========================================================

class EvaluationError(Exception):
    """
    Raised when evaluation fails.
    """
    pass


# ==========================================================
# Prediction Result
# ==========================================================

@dataclass(frozen=True)
class PredictionResult:
    """
    Raw predictions produced by the model.
    """

    loss: float

    predictions: np.ndarray

    labels: np.ndarray


# ==========================================================
# Evaluation Result
# ==========================================================

@dataclass(frozen=True)
class EvaluationResult:
    """
    Complete evaluation metrics.
    """

    samples: int

    loss: float

    accuracy: float

    precision: float

    recall: float

    f1: float

    predictions: np.ndarray

    labels: np.ndarray


# ==========================================================
# Evaluator
# ==========================================================

class Evaluator:
    """
    Responsible for model evaluation.

    Responsibilities
    ----------------
    • Run inference
    • Collect predictions
    • Compute evaluation metrics

    This class intentionally does NOT

    • Train models
    • Update optimizer
    • Update scheduler
    • Save checkpoints
    """

    # ------------------------------------------------------
    # Constructor
    # ------------------------------------------------------

    def __init__(
        self,
        model: torch.nn.Module,
        dataloader: DataLoader,
        context: ExperimentContext,
    ):

        self.model = model

        self.dataloader = dataloader

        self.context = context

        self.device = context.device

    # ------------------------------------------------------
    # Helpers
    # ------------------------------------------------------

    def _move_batch_to_device(
        self,
        batch: dict,
    ) -> dict:
        """
        Move an entire batch onto the target device.
        """

        return {

            key: value.to(self.device)

            for key, value in batch.items()

        }    # ------------------------------------------------------
    # Prediction
    # ------------------------------------------------------

    def predict(
        self,
    ) -> PredictionResult:
        """
        Run inference on the configured dataloader.

        Returns
        -------
        PredictionResult
            Average loss, predictions and labels.
        """

        self.model.eval()

        total_loss = 0.0

        predictions = []

        labels = []

        progress_bar = tqdm(
            self.dataloader,
            desc="Evaluating",
            leave=False,
        )

        with torch.no_grad():

            for batch in progress_bar:

                # ------------------------------------------
                # Move Batch
                # ------------------------------------------

                batch = self._move_batch_to_device(
                    batch,
                )

                # ------------------------------------------
                # Forward Pass
                # ------------------------------------------

                outputs = self.model(
                    **batch,
                )

                loss = outputs.loss

                logits = outputs.logits

                # ------------------------------------------
                # Statistics
                # ------------------------------------------

                total_loss += loss.detach().item()

                batch_predictions = torch.argmax(
                    logits,
                    dim=-1,
                )

                predictions.extend(
                    batch_predictions.cpu().numpy()
                )

                labels.extend(
                    batch["labels"].cpu().numpy()
                )

                progress_bar.set_postfix(
                    loss=f"{loss:.4f}",
                )

        average_loss = (
            total_loss
            / len(self.dataloader)
        )

        return PredictionResult(
            loss=average_loss,
            predictions=np.asarray(predictions),
            labels=np.asarray(labels),
        )    # ------------------------------------------------------
    # Evaluation
    # ------------------------------------------------------

    def evaluate(
        self,
    ) -> EvaluationResult:
        """
        Evaluate the model on the configured dataset.

        Returns
        -------
        EvaluationResult
            Complete evaluation metrics.
        """

        # ----------------------------------------------
        # Run Inference
        # ----------------------------------------------

        prediction_result = self.predict()

        # ----------------------------------------------
        # Compute Metrics
        # ----------------------------------------------

        accuracy = accuracy_score(
            prediction_result.labels,
            prediction_result.predictions,
        )

        precision, recall, f1, _ = (
            precision_recall_fscore_support(
                prediction_result.labels,
                prediction_result.predictions,
                average="weighted",
                zero_division=0,
            )
        )

        # ----------------------------------------------
        # Package Results
        # ----------------------------------------------

        return EvaluationResult(
            samples=len(
                prediction_result.labels,
            ),
            loss=prediction_result.loss,
            accuracy=float(accuracy),
            precision=float(precision),
            recall=float(recall),
            f1=float(f1),
            predictions=prediction_result.predictions,
            labels=prediction_result.labels,
        )    # ------------------------------------------------------
    # Summary
    # ------------------------------------------------------

    @staticmethod
    def print_summary(
        result: EvaluationResult,
    ) -> None:
        """
        Print evaluation summary.
        """

        print("=" * 70)
        print("EVALUATION SUMMARY")
        print("=" * 70)

        print(
            f"Samples   : "
            f"{result.samples:,}"
        )

        print(
            f"Loss      : "
            f"{result.loss:.4f}"
        )

        print(
            f"Accuracy  : "
            f"{result.accuracy:.4f}"
        )

        print(
            f"Precision : "
            f"{result.precision:.4f}"
        )

        print(
            f"Recall    : "
            f"{result.recall:.4f}"
        )

        print(
            f"F1 Score  : "
            f"{result.f1:.4f}"
        )

        print("=" * 70)

## EXPERIMENT RUNNER

In [29]:
# ==========================================================
# Experiment Runner
# ==========================================================

from dataclasses import dataclass

import time


# ==========================================================
# Experiment Exceptions
# ==========================================================

class ExperimentError(Exception):
    """
    Raised when experiment execution fails.
    """
    pass


# ==========================================================
# Experiment Result
# ==========================================================

@dataclass(frozen=True)
class ExperimentResult:
    """
    Complete experiment outputs.
    """

    history: TrainingHistory

    evaluation: EvaluationResult

    runtime_seconds: float


# ==========================================================
# Experiment Runner
# ==========================================================

class ExperimentRunner:
    """
    Executes a complete experiment from configuration
    to final evaluation.
    """

    # ------------------------------------------------------
    # Constructor
    # ------------------------------------------------------

    def __init__(
        self,
        config: ExperimentConfig,
    ):

        self.config = config    # ------------------------------------------------------
    # Run Experiment
    # ------------------------------------------------------

    def run(
        self,
    ) -> ExperimentResult:
        """
        Execute the complete experiment pipeline.
        """

        # ----------------------------------------------
        # Start Timer
        # ----------------------------------------------

        start_time = time.perf_counter()

        # ----------------------------------------------
        # Validate Configuration
        # ----------------------------------------------

        ConfigurationValidator.validate(
            self.config,
        )

        # ----------------------------------------------
        # Prepare Runtime
        # ----------------------------------------------

        SeedManager.set_seed(
            self.config.experiment.seed,
        )

        device = DeviceManager.get_device(
            self.config.runtime.device,
        )

        runtime_info = DeviceManager.get_device_info(
            device,
        )

        paths = DirectoryManager.prepare(
            self.config,
        )

        context = ExperimentContext(
            config=self.config,
            paths=paths,
            device=device,
            runtime_info=runtime_info,
            start_time=datetime.now(),
        )

        # ----------------------------------------------
        # Load Datasets
        # ----------------------------------------------

        (
            train_ds,
            valid_ds,
            test_ds,
        ) = DatasetManager.load_all(
            context,
        )

        # ----------------------------------------------
        # Prepare Tokenizer & DataLoaders
        # ----------------------------------------------

        (
            tokenizer,
            train_loader,
            valid_loader,
            test_loader,
        ) = TokenizerManager.prepare(
            train_ds,
            valid_ds,
            test_ds,
            context,
        )

        # ----------------------------------------------
        # Build Model
        # ----------------------------------------------

        model_bundle = ModelFactory.build(
            context,
        )

        model = model_bundle.model

        model.to(device)

        # ----------------------------------------------
        # Build Training Utilities
        # ----------------------------------------------

        training_bundle = (
            TrainingManager.prepare(
                model=model,
                train_loader=train_loader,
                context=context,
            )
        )

        # ----------------------------------------------
        # Train Model
        # ----------------------------------------------

        trainer = Trainer(
            model=model,
            training=training_bundle,
            train_loader=train_loader,
            validation_loader=valid_loader,
            context=context,
        )

        history = trainer.fit()

        # ----------------------------------------------
        # Evaluate Model
        # ----------------------------------------------

        evaluator = Evaluator(
            model=model,
            dataloader=test_loader,
            context=context,
        )

        evaluation = evaluator.evaluate()

        # ----------------------------------------------
        # Runtime
        # ----------------------------------------------

        runtime_seconds = (
            time.perf_counter()
            - start_time
        )

        # ----------------------------------------------
        # Return Results
        # ----------------------------------------------

        return ExperimentResult(
            history=history,
            evaluation=evaluation,
            runtime_seconds=runtime_seconds,
        )    # ------------------------------------------------------
    # Summary
    # ------------------------------------------------------

    @staticmethod
    def print_summary(
        result: ExperimentResult,
    ) -> None:
        """
        Print experiment summary.
        """

        print("=" * 70)
        print("EXPERIMENT SUMMARY")
        print("=" * 70)

        print(
            f"Runtime (sec) : "
            f"{result.runtime_seconds:.2f}"
        )

        print(
            f"Train Epochs  : "
            f"{len(result.history.train_loss)}"
        )

        print(
            f"Final Train Loss : "
            f"{result.history.train_loss[-1]:.4f}"
        )

        print(
            f"Final Valid Loss : "
            f"{result.history.validation_loss[-1]:.4f}"
        )

        print(
            f"Test Accuracy : "
            f"{result.evaluation.accuracy:.4f}"
        )

        print(
            f"Test Precision: "
            f"{result.evaluation.precision:.4f}"
        )

        print(
            f"Test Recall   : "
            f"{result.evaluation.recall:.4f}"
        )

        print(
            f"Test F1 Score : "
            f"{result.evaluation.f1:.4f}"
        )

        print("=" * 70)

In [30]:
# ==========================================================
# Single Experiment (FFT Baseline)
# ==========================================================

from pathlib import Path

# ----------------------------------------------------------
# Dataset Paths
# ----------------------------------------------------------

DATASET_ROOT = Path(
    "/kaggle/input/datasets/yashjaiswal04518/indicxnli-preprocessed-v1/hi"
)

# ----------------------------------------------------------
# Build Configuration
# ----------------------------------------------------------

config = ExperimentManager.create(

    experiment_name="FFT_Hindi_100",

    description="Full Fine-Tuning baseline",

    language=Language.HINDI,

    budget="500",

    method=PEFTMethod.FFT,

    train_path=DATASET_ROOT / "train_100.parquet",

    validation_path=DATASET_ROOT / "valid.parquet",

    test_path=DATASET_ROOT / "test.parquet",

    output_dir=Path("/kaggle/working/experiments"),

    epochs=20,

    batch_size=4,

    learning_rate=2e-5,

)

# ----------------------------------------------------------
# Execute Experiment
# ----------------------------------------------------------

runner = ExperimentRunner(config)

result = runner.run()

# ----------------------------------------------------------
# Print Results
# ----------------------------------------------------------

print()

ExperimentRunner.print_summary(result)

print()

print("Training Loss History")
print("-" * 70)
print(result.history.train_loss)

print()

print("Validation Loss History")
print("-" * 70)
print(result.history.validation_loss)

print()

Evaluator.print_summary(
    result.evaluation,
)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizing dataset:   0%|          | 0/100 [00:00<?, ? examples/s]

...

Training:   0%|          | 0/25 [00:00<?, ?it/s]

Validation:   0%|          | 0/623 [00:00<?, ?it/s]

Train Loss      : 0.0270
Validation Loss : 2.6816
Time            : 12.99 sec

Training Complete


Evaluating:   0%|          | 0/1253 [00:00<?, ?it/s]


EXPERIMENT SUMMARY
Runtime (sec) : 301.33
Train Epochs  : 20
Final Train Loss : 0.0270
Final Valid Loss : 2.6816
Test Accuracy : 0.3754
Test Precision: 0.3763
Test Recall   : 0.3754
Test F1 Score : 0.3756

Training Loss History
----------------------------------------------------------------------
[1.1133642578125, 1.11138671875, 1.10885498046875, 1.10930908203125, 1.09859375, 1.047529296875, 0.9924462890625, 0.82298095703125, 0.5351171875, 0.3433587646484375, 0.2152642822265625, 0.09954429626464843, 0.04528541564941406, 0.04231979370117187, 0.035635337829589844, 0.019605865478515627, 0.026914443969726563, 0.019223136901855467, 0.030702838897705077, 0.026982421875]

Validation Loss History
----------------------------------------------------------------------
[1.1072833457689606, 1.1004299696528892, 1.0983976850922954, 1.0995392286567014, 1.115466367375602, 1.0991930035488564, 1.1312785875739868, 1.2072941518327398, 1.41348403759217, 1.791506747372843, 1.843393212528137, 2.13499287502